In [1]:
import os
import torch
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

MAX_LENGTH = 256
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

save_dir = os.path.abspath("models/finetuned_distilbert")
finetuned_model = DistilBertForSequenceClassification.from_pretrained(
    save_dir,
    num_labels=1,
).to(device)
tokenizer = DistilBertTokenizerFast.from_pretrained(save_dir)
print(f"Loaded model from {save_dir} on {device}")

/Users/dj/Desktop/School/CS4964/CS4964-Senitment-Analysis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/dj/Desktop/School/CS4964/CS4964-Senitment-Analysis/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded model from /Users/dj/Desktop/School/CS4964/CS4964-Senitment-Analysis/models/finetuned_distilbert on cpu


In [34]:
# real world review we found
# given 0.5/5 (lowest possible on that site) stars -> 1/10
movie_review = "worst movie of the year… straight up headache and chest pain inducing, we swear."
# given 5 stars -> 10/10
restaurant_review = "The food is terrific. I had the Sparta and it was delicious. Great ambience and food quality is high."
# 4/5 start -> 8/10
book_review = '''About dang time the main characters are POC in a romantasy!! Loved the MC’s dynamics! This was enjoyable but it was predictable and lacked depth at times.

I tried so hard, I really did to ignore it, but Thane was the name of the kid who mercilessly bullied me in school…'''

In [35]:
for review in [movie_review, restaurant_review, book_review]:
    movie_encoding = tokenizer(
        review,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).to(device)

    finetuned_model.eval()
    with torch.no_grad():
        outputs = finetuned_model(**movie_encoding)
        result = outputs.logits.squeeze().item() * 10
        print(f"Review: {review}")
        print(f"Predicted rating (0-10): {result:.2f}")

Review: worst movie of the year… straight up headache and chest pain inducing, we swear.
Predicted rating (0-10): 0.15
Review: The food is terrific. I had the Sparta and it was delicious. Great ambience and food quality is high.
Predicted rating (0-10): 9.83
Review: About dang time the main characters are POC in a romantasy!! Loved the MC’s dynamics! This was enjoyable but it was predictable and lacked depth at times.

I tried so hard, I really did to ignore it, but Thane was the name of the kid who mercilessly bullied me in school…
Predicted rating (0-10): 6.63
